## 1. Classification metrics — corrections

### Q1. When is recall more important than precision?

- Recall measures how many actual positive cases the model successfully identifies.
- Low recall means the model misses many real positive cases (false negatives).
- In disease screening, missing a potential case may be more dangerous than flagging someone who turns out not to have the disease.

$[ \text{Recall}=\frac{TP}{TP+FN} $]

### Q2. When is precision more important than recall?

- Precision measures how many predicted positives are actually positive.
- Low precision means too many false positives.
- A human-review step can handle uncertain cases, but whether a false negative is acceptable depends on the business rules and the cost of missing a legitimate claim.

$[ \text{Precision}=\frac{TP}{TP+FP} $]

Remember: the right metric depends on the cost of each type of mistake, not just the industry.

### Q3. Why can accuracy be misleading on imbalanced datasets?

Suppose a fraud dataset contains 1,000 transactions:

- 990 legitimate transactions
- 10 fraudulent transactions

A model predicts every transaction as legitimate.

| Metric          | Result                                                                 |
| --------------- | ---------------------------------------------------------------------- |
| Accuracy        | 99%                                                                    |
| Fraud recall    | 0%                                                                     |
| Fraud precision | Undefined in this example; scikit-learn generally reports 0 by default |

The model achieves 99% accuracy while detecting zero fraud cases.

That's the problem: accuracy can hide poor performance on a rare but important class.

### Q4. ROC-AUC vs PR-AUC

### ROC-AUC

ROC plots the true positive rate (recall) against the false positive rate at different classification thresholds.

It measures how well the model separates the classes across thresholds. It can be useful when evaluating both positive and negative classes matters.

### PR-AUC

The precision–recall curve plots precision against recall across different thresholds.

It focuses on the positive class and is often more informative when positive cases are rare and false positives matter.

Interview takeaway: On a heavily imbalanced fraud or disease-detection dataset, inspect PR-AUC alongside recall, precision, and the confusion matrix. ROC-AUC is still useful, but can look reassuring even when positive-class performance is poor.

One subtlety: PR-AUC depends on the prevalence of the positive class, so compare it in the context of the dataset's class distribution.

### Q5. Why should SMOTE be applied only to training data?

SMOTE creates synthetic minority-class examples using existing minority-class samples and their neighbours.

If you apply it before splitting the dataset:

1. Synthetic samples may be influenced by observations that later enter your test set.
2. Related or near-duplicate information can leak across the split.
3. Your test results may look better than real-world performance warrants.

The correct order is:

1. Split the original data into training and test sets.
2. Apply SMOTE only to the training data.
3. Train the model on the resampled training data.
4. Evaluate on the untouched test data, preserving its original class distribution.

In cross-validation, put SMOTE inside an `imblearn` pipeline so it runs only on each fold's training portion.

### The three datasets

Training set :
> Used to fit the model's parameters — for example, the weights in Logistic Regression or the split rules in a Decision Tree.

Validation set :
> Used to compare models, tune hyperparameters, and choose classification thresholds. It helps you make decisions without using the test set.

Test set :
> Used for the final, relatively unbiased evaluation after model and hyperparameter choices are settled. Don't keep checking it to decide what to change.

In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=10_000,
    n_features=15,
    weights=[0.9, 0.1],
    random_state=42
)

# First: separate the final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=42
)

# Second: separate validation from the remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.15 / 0.85,
    stratify=y_train_val,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (7000, 15) (7000,)
Validation: (1500, 15) (1500,)
Test: (1500, 15) (1500,)


## 3. Cross-validation — making better use of data

A single train/validation split can give a misleading impression if the particular split happens to be unusually easy or difficult.

K-fold cross-validation addresses this by splitting the training data into $(K$) folds.


## 4. Overfitting, underfitting, and the bias–variance trade-off

These concepts explain why a model performs well during training but fails on new data.

[Simplifying Regularization Part 1 Ridge to Elastic Net | DigitalOcean]

<img src="./data/underfitting.jpg" width="50"> Underfitting — model too simple 

Training performance is poor, and validation performance is poor too.

Example: using a straight line when the true relationship is strongly nonlinear.

Try: richer features, a more flexible model, or weaker regularization.

[Generalization, Overfitting, and Under-fitting in Supervised Learning | by Ivan Zakharchuk | Medium]

<img src="./data/goodFit.jpg" width="50"> Good generalization

Training and validation performance are both good, with a reasonably small gap.

Goal: strong performance on unseen data, not simply a perfect training score.

[Why Your ML Model Fails in the Real World (and How to Fix It!) | by DSiPE - Data Science in Plain English | Medium]

<img src="./data/overfitting.jpg" width="50"> Overfitting — model too complex

Training performance is excellent, but validation performance is substantially worse.

Example: a deep Decision Tree memorizing noise in customer records.

Try: regularization, a simpler model, more representative data, or better feature selection.

### Bias vs variance

- High bias: the model makes overly simplistic assumptions and misses real patterns. Often associated with underfitting.
- High variance: the model is too sensitive to the particular training sample. Often associated with overfitting.

The trade-off is about finding a model complex enough to capture useful patterns without learning noise.

One important nuance: training–validation gaps are diagnostic clues, not definitive proof. Data distribution shifts, noisy labels, and small datasets can also explain poor validation results.


## What is feature engineering?
> Feature engineering is the process of transforming raw data into useful input variables that help a machine learning model learn patterns.

Imagine you're building a customer churn prediction model for a subscription business.

Your raw dataset looks like this:

| Customer | Signup date | Last activity | Monthly bill |
| -------- | ----------- | ------------- | ------------ |
| A        | Jan 2025    | Oct 2026      | ₹500         |
| B        | Mar 2026    | Apr 2026      | ₹900         |
| C        | Jun 2025    | Oct 2026      | ₹700         |

The raw columns are useful, but we can extract more meaningful features from them.

Raw data

Signup date · Last activity · Monthly bill

Feature engineering

Calculate tenure · Days since activity · Annualized bill

Model-ready features

Customer tenure · Inactivity days · Annualized bill

The model can now learn from more meaningful representations of the customer.

## Common feature engineering techniques

A. Date and time features

- Extract year, month, day of week, customer age, or days since last activity.
- Example: `days_since_last_login` may be more predictive of churn than the raw timestamp.

B. Mathematical transformations

- Create ratios, differences, totals, and averages.
- Example: `claim_amount / policy_limit`, or average monthly spending.

C. Grouping and aggregation

- Summarize historical behavior, such as the number of claims per customer or average transactions per month.
- Be careful to use only information available at prediction time.

D. Interaction and nonlinear features - 
- Combine variables when their relationship may matter.
- Example: `monthly_bill * contract_length`, or a squared feature such as `age ** 2`.

## What is feature selection?

Feature engineering creates or transforms features. Feature selection chooses which existing or engineered features to retain.

Imagine you have 200 columns, but many are redundant, irrelevant, or noisy. Using all of them might increase computation, complicate the model, and sometimes hurt generalization.

Three common approaches:

### A. Filter methods

Select features using statistical properties before fitting the final model.

Examples: correlation, mutual information, chi-square tests.

Useful for quickly narrowing a large set of candidate features. Some methods detect only particular kinds of relationships.

### B. Wrapper methods

Evaluate subsets of features by repeatedly fitting a model.

Example: Recursive Feature Elimination (RFE).

Can find useful combinations for a particular estimator, but may be computationally expensive.

### C. Embedded methods

Feature selection happens as part of model training.

Examples: Lasso regression, L1-regularized Logistic Regression, tree-based feature importance.

Often practical, but the selected features depend on the model and data. Tree importance can favor certain feature types or split patterns.

In [8]:
# Feature selection
import pandas as pd
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif
)

df = pd.DataFrame({
    "customer_id": [101, 102, 103, 104],
    "tenure_months": [24, 3, 18, 6],
    "monthly_bill": [500, 900, 700, 400],
    "total_logins": [120, 5, 80, 12],
    "days_since_last_login": [2, 65, 7, 40],
    "churn": [0, 1, 0, 1]
})


X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

selector = SelectKBest(
    score_func=mutual_info_classif,
    k=3
)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("Selected features:", selected_features.tolist())

Selected features: ['monthly_bill', 'total_logins', 'days_since_last_login']
